# Models: Parametric vs DML

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import sys
from pathlib import Path
import statsmodels.api as sm

PROJECT_ROOT = Path.cwd().parent  # assumes notebooks/ is one level below root
sys.path.append(str(PROJECT_ROOT))

from src.load_data import load_feature

In [2]:
df = load_feature()
df.head()

,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,accommodates,bathrooms,...,amenity_Dedicated workspace,amenity_Toaster,amenity_Freezer,amenity_Shower gel,amenity_First aid kit,amenity_Dining table,amenity_Cleaning products,amenity_Self check-in,amenity_Fire extinguisher,amenity_Long term stays allowed
0,1.00,0.96,1,1.098612,1.791759,2,1,1,1,1.0,...,1,1,1,1,1,1,0,1,1,1
1,0.88,0.88,1,1.386294,2.833213,3,1,1,6,2.0,...,1,1,1,0,0,1,1,0,0,1
2,1.00,0.98,0,1.386294,4.691348,2,1,1,4,1.0,...,0,0,0,0,0,0,0,0,0,0
3,1.00,0.91,0,0.693147,0.693147,3,1,1,5,1.5,...,1,0,0,0,0,0,0,1,1,0
4,1.00,1.00,1,1.098612,1.609438,2,1,1,2,0.0,...,1,0,0,0,1,0,0,1,1,0


In [3]:
df.dtypes.value_counts()

bool       131
int64       46
float64     20
Name: count, dtype: int64

In [4]:
df.select_dtypes(include="object").columns.tolist()

[]

## Parametric model

In [5]:
#first spec without borough
borough_columns = [i for i in df.columns if "borough" in i ]

In [6]:
#outcome and treatment variable
y = df["log_price"]
d =df["log_rivals_500m"]

#exclude and controls
exclude = {"log_price", "log_rivals_500m", "loc_fe", "log_rivals_c_sq", "log_rivals_c"}
X = df[[c for c in df.columns if (c not in exclude) & (c not in borough_columns) ]]


#controls and treatment
Z = pd.concat([d,X], axis=1)

#adding constant
Z = sm.add_constant(Z, has_constant="add")

#changing data types
y = y.astype(float)
Z = Z.astype(float)

Z.shape



(42898, 162)

In [7]:
#running model
model = sm.OLS(y, Z).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["loc_fe"]}
)

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.724
Model:                            OLS   Adj. R-squared:                  0.723
Method:                 Least Squares   F-statistic:                     7938.
Date:                Thu, 26 Feb 2026   Prob (F-statistic):          7.30e-162
Time:                        14:52:12   Log-Likelihood:                -20898.
No. Observations:               42898   AIC:                         4.212e+04
Df Residuals:                   42736   BIC:                         4.352e+04
Df Model:                         161                                         
Covariance Type:              cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

c:\Users\danil\anaconda3\envs\vair312\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 161, but rank is 62
  warnings.warn('covariance of constraints does not have full '


In [8]:
beta = model.params["log_rivals_500m"]
se   = model.bse["log_rivals_500m"]
ci_l, ci_u = model.conf_int().loc["log_rivals_500m"].tolist()

print(beta, se, ci_l, ci_u)

0.008654797008828867 0.01297476530876411 -0.016775275704208503 0.03408486972186624


### Non linear treatment

In [9]:
#outcome and treatment variable
y_nl = df["log_price"]
d_nl =df["log_rivals_c"]

#exclude and controls
exclude_nl = {"log_price", "log_rivals_500m", "loc_fe", "log_rivals_c"}
X_nl = df[[c for c in df.columns if (c not in exclude_nl) & (c not in borough_columns) ]]


#controls and treatment
Z_nl = pd.concat([d_nl,X_nl], axis=1)

#adding constant
Z_nl = sm.add_constant(Z_nl, has_constant="add")

#changing data types
y_nl = y_nl.astype(float)
Z_nl = Z_nl.astype(float)

Z_nl.shape



(42898, 163)

In [10]:
#running model
model_nl = sm.OLS(y_nl, Z_nl).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["loc_fe"]}
)

print(model_nl.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.724
Model:                            OLS   Adj. R-squared:                  0.723
Method:                 Least Squares   F-statistic:                     7259.
Date:                Thu, 26 Feb 2026   Prob (F-statistic):          4.48e-160
Time:                        14:52:12   Log-Likelihood:                -20885.
No. Observations:               42898   AIC:                         4.210e+04
Df Residuals:                   42735   BIC:                         4.351e+04
Df Model:                         162                                         
Covariance Type:              cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

c:\Users\danil\anaconda3\envs\vair312\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 162, but rank is 63
  warnings.warn('covariance of constraints does not have full '


### Double Lasso PDS

In [11]:
#setting up controls, treatment adn outcome

y_lasso = "log_price"
d_lasso = "log_rivals_500m"
fe_lasso = "loc_fe"

#fe columns
fe_cols = [c for c in df.columns if c.startswith("fe_")]

exclude = {y_lasso, d_lasso, "rivals_500m", "price", "id",
            fe_lasso, "log_rivals_c_sq", "log_rivals_c"}

# Candidate controls for selection (exclude FE dummies from selection step)
X_cand_cols = [c for c in df.columns if c not in exclude and c not in fe_cols]

# FE dummies forced-in at the final stage
X_force_cols = fe_cols

In [ ]:
#Variabels for lasso
X_cand = df[X_cand_cols].values
Y = df[y_lasso].values
T = df[d_lasso].values

# ElasticNetCV with l1_ratio = 1 for lasso
enet = ElasticNetCV(
    l1_ratio=1.0,
    alphas=None,
    cv=5,
    random_state=0,
    max_iter=10000
)

# Pipeline: standardize then fit
y_selector = Pipeline([("scaler", StandardScaler()), ("enet", enet)])
t_selector = Pipeline([("scaler", StandardScaler()), ("enet", enet)])

y_selector.fit(X_cand, Y)
t_selector.fit(X_cand, T)

coef_y = y_selector.named_steps["enet"].coef_
coef_t = t_selector.named_steps["enet"].coef_

S_y = set(np.array(X_cand_cols)[coef_y != 0])
S_t = set(np.array(X_cand_cols)[coef_t != 0])
S_union = sorted(list(S_y.union(S_t)))

print("Selected for Y:", len(S_y))
print("Selected for T:", len(S_t))
print("Union selected:", len(S_union))
